# Redacted Demo: LLM Phrase Classification

This notebook contains only the publishable method steps for clinician free-text phrase classification.

It intentionally excludes private data curation, scraping, patient linking, and internal workflow code.

## 1) Setup

Run from inside this repository and make sure dependencies are installed from `requirements.txt`.

In [ ]:
from pathlib import Path
import os
import sys
import sqlite3
import pandas as pd

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

assert (repo_root / "src").exists(), "Could not find repository root with src/"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

## 2) Configure inference run

The same pipeline supports rule-based, local llama.cpp, and OpenAI-compatible backends.

- `MODEL = "rules"` for deterministic baseline
- `MODEL = "local_llm"` for llama.cpp + Qwen
- `MODEL = "gpt-4o-mini"` (or another model id) for OpenAI backend

In [ ]:
DB_PATH = repo_root / "data" / "app" / "romeu_unknown_phrases.sqlite"
RUN_NAME = "qwen_local_demo_v1"
MODEL = "local_llm"
PROMPT_VERSION = "v1"
LIMIT = 50

SYSTEM_PROMPT = (
    "You are Qwen3-30B-A3B-Thinking, a helpful reasoning assistant for "
    "clinical oral medicine. Think step by step, but FINAL output must be a "
    "single JSON object with keys: category, confidence, rationale."
)

print("DB exists:", DB_PATH.exists())
print("Model backend:", MODEL)

## 3) Backend environment variables

For local llama-server (recommended for speed, avoids model reload each call), set:

- `LOCAL_LLM_SERVER_URL=http://127.0.0.1:8080/completion`

If you do not use server mode, set:

- `LOCAL_LLM_BIN=/path/to/llama-cli`
- `LOCAL_LLM_MODEL=/path/to/model.gguf`

In [ ]:
# Example: local server endpoint
os.environ.setdefault("LOCAL_LLM_SERVER_URL", "http://127.0.0.1:8080/completion")

# Optional CLI fallback variables (only needed if LOCAL_LLM_SERVER_URL is not used):
# os.environ["LOCAL_LLM_BIN"] = "/path/to/llama-cli"
# os.environ["LOCAL_LLM_MODEL"] = "/path/to/Qwen3-30B-A3B-Thinking.gguf"

print("LOCAL_LLM_SERVER_URL:", os.getenv("LOCAL_LLM_SERVER_URL"))

## 4) Run inference

This stores predictions in `phrase_prediction` and metadata in `model_run`.

In [ ]:
from src.inference.phrase_classifier import run_inference

inserted, total = run_inference(
    db_path=str(DB_PATH),
    run_name=RUN_NAME,
    model=MODEL,
    prompt_version=PROMPT_VERSION,
    limit=LIMIT,
    log_every=10,
    verbose=True,
    debug_stream=False,
    system_prompt=SYSTEM_PROMPT,
)

print({"inserted": inserted, "scanned": total})

## 5) Inspect latest run

Quick sanity checks for category distribution and sample predictions.

In [ ]:
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
conn.row_factory = sqlite3.Row

run_row = conn.execute(
    "SELECT id, name, model, prompt_version, created_at FROM model_run ORDER BY created_at DESC LIMIT 1"
).fetchone()

if run_row is None:
    raise RuntimeError("No model_run rows found. Run inference first.")

run_id = int(run_row["id"])
print(dict(run_row))

dist_df = pd.read_sql_query(
    """
    SELECT category, COUNT(*) AS n, ROUND(AVG(confidence), 4) AS avg_conf
    FROM phrase_prediction
    WHERE run_id = ? AND status = 'ok'
    GROUP BY category
    ORDER BY n DESC
    """,
    conn,
    params=(run_id,),
)
display(dist_df)

sample_df = pd.read_sql_query(
    """
    SELECT rg, phrase_id, phrase_text, category, confidence, rationale
    FROM phrase_prediction
    WHERE run_id = ?
    ORDER BY confidence DESC
    LIMIT 20
    """,
    conn,
    params=(run_id,),
)
display(sample_df)

conn.close()

## Notes

- Full protocol (prompt, decoding parameters, and fallback behavior): `docs/llm_phrase_classification_protocol.md`
- Core implementation: `src/inference/phrase_classifier.py`
- CLI entrypoint: `scripts/inference/run_phrase_classifier.py`